# Verify the video timing QC change in the batch pipeline

Library release `v0.1.0` replaces the timing QC inside
`integrate_keypoints_with_video_time`: keypoint `time_raw` now comes from
Harp time corrected for dropped frames and Harp glitches, and CSVs with a
header row load instead of crashing. This notebook runs `run_batch_analysis`
on a few attached sessions twice, once with the library just before the QC
(tag `pre-video-timing-qc`) and once with the QC (tag `v0.1.0`, or any later
ref set in `NEW_REF`), and diffs every intermediate parquet, to catch
anything that changes unexpectedly.

Steps:

1. Clone both library versions into scratch. Each run imports its version
   through `PYTHONPATH` in a subprocess, so the capsule's installed library
   is not touched.
2. Build the prediction-CSV list for every attached session with raw data
   and a `videoprocessed` asset (saved to scratch).
3. Classify every session's bottom-camera timing with `video_timing_qc`, and
   pick a few sessions to run: up to 2 per problem class (glitch, drops,
   ...), up to 2 header-row CSVs, and 2 ok ones. The scan reads every
   session's CSV; only the chosen few go through the batch.
4. Run the batch with each version, then diff (`scripts/compare_batch_outputs.py`).

What to expect:

Sessions are grouped by what the new correction will do (`timing_action`
below, from `vtq.timing_verdict` and the checks) and which checks failed. The new run uses each
session's Harp trigger log (`Event_94.bin`) when it exists, the library's
default; for sessions without drops that gives the same times as the CSV.

| Action | Expected diff |
|---|---|
| `use harp as written` | every file `identical`, including sessions whose frame numbers / camera time are corrupted (both versions use Harp as written) |
| `fix glitches` | tiny differences in time columns (the old QC also interpolated glitches, and nudged the next row by µs) |
| `refuse: ...` | `missing in new` (e.g. `refuse: harp_evenly_spaced` for a Harp clock step) |
| `re-index` | large differences in time and everything matched by time |
| header-row CSV | `missing in old`: `main` crashes on these files |

A session that fails in *both* runs for an unrelated reason (e.g. an NWB
assertion) has no files to compare; it is flagged `CHECK` and its errors
are in the run logs printed in section 7.

None of the attached sessions has dropped frames (checked from the CSVs on
S3, 2026-09-29), so the drop case is covered by the library's
`examples/video_timing_qc_validation.ipynb` instead.

Everything is written under `/root/capsule/scratch/video_timing_qc_test/`.
Code-Ocean-only, guarded by `IS_CO`. Run top to bottom in a fresh kernel.

## 1. Setup

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

if Path("/root/capsule").exists():
    DATA = Path("/root/capsule/data")
    WORK = Path("/root/capsule/scratch/video_timing_qc_test")
    WORK.mkdir(parents=True, exist_ok=True)
    IS_CO = True
else:
    DATA = WORK = None
    IS_CO = False
    print("Local environment: nothing to run here. Run this notebook on Code Ocean.")

LIBRARY_URL = "https://github.com/AllenNeuralDynamics/aind-dynamic-foraging-behavior-video-analysis.git"
OLD_REF = "pre-video-timing-qc"   # library just before the QC (tag)
NEW_REF = "v0.1.0"                # library with the QC (tag, branch or later release)

N_OK = 2            # ok sessions to run
N_PER_CLASS = 2     # max sessions per other class (glitch, header row, ...)
RUN_CSV_ONLY = False     # also run the new version with use_trigger_log=False
REFRESH_NEW = False      # True: re-clone NEW_REF, redo the timing scan and the new run

## 2. Clone both library versions

Shallow clones into scratch, made once. After changing `NEW_REF`, set
`REFRESH_NEW = True` (re-clones it and redoes the timing scan and the new
run; the old run is kept).

In [ ]:
def clone(ref, folder):
    """Shallow-clone the library at `ref` into `folder`, once; return its commit."""
    if not folder.exists():
        subprocess.run(
            ["git", "clone", "--quiet", "--depth", "1", "--branch", ref, LIBRARY_URL, str(folder)],
            check=True,
        )
    return subprocess.run(
        ["git", "-C", str(folder), "rev-parse", "--short", "HEAD"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()


if IS_CO:
    LIB_OLD = WORK / "lib_old"
    LIB_NEW = WORK / "lib_new"
    if REFRESH_NEW:
        import shutil
        for stale in [LIB_NEW, WORK / "out_new", WORK / "out_csv_only"]:
            shutil.rmtree(stale, ignore_errors=True)
        (WORK / "timing_scan.csv").unlink(missing_ok=True)
    print(f"old: {OLD_REF} @ {clone(OLD_REF, LIB_OLD)}")
    print(f"new: {NEW_REF} @ {clone(NEW_REF, LIB_NEW)}")
    assert not (LIB_OLD / "src/aind_dynamic_foraging_behavior_video_analysis/video_timing_qc.py").exists(), \
        "OLD_REF already has video_timing_qc; it should be the version before it"
    assert (LIB_NEW / "src/aind_dynamic_foraging_behavior_video_analysis/video_timing_qc.py").exists()

The rest of this kernel uses the **new** library (for `video_timing_qc` and
the session lookup helpers). It must come first on `sys.path` before anything
imports the library, hence the fresh kernel.

In [ ]:
if IS_CO:
    sys.path.insert(0, str(LIB_NEW / "src"))
    import aind_dynamic_foraging_behavior_video_analysis as lib
    from aind_dynamic_foraging_behavior_video_analysis import video_timing_qc as vtq
    from aind_dynamic_foraging_behavior_video_analysis.kinematics.tongue_kinematics_utils import (
        find_behavior_videos_folder,
        find_video_csv_path,
    )

    assert Path(lib.__file__).is_relative_to(LIB_NEW), lib.__file__
    print("kernel library:", lib.__file__)

## 3. Prediction CSVs for every attached session

One per session: the first `pred_outputs/video_preds/*predictions.csv` across
that session's `videoprocessed` assets. Which model it is does not matter
here, because both runs use the same file. The full list is saved to
`pred_csv_list.json`.

In [ ]:
if IS_CO:
    raw_sessions = sorted(
        p.name for p in DATA.iterdir()
        if p.is_dir() and p.name.startswith("behavior_") and p.name.count("_") == 3
    )
    pred_csvs = {}
    for session in raw_sessions:
        candidates = sorted(DATA.glob(f"{session}_videoprocessed*/pred_outputs/video_preds/*predictions.csv"))
        if candidates:
            pred_csvs[session] = str(candidates[0])

    (WORK / "pred_csv_list.json").write_text(json.dumps(list(pred_csvs.values()), indent=2))
    print(f"{len(raw_sessions)} raw sessions attached, {len(pred_csvs)} with predictions")
    print("saved", WORK / "pred_csv_list.json")

## 4. Check each session's video timing

Runs `check_video_timing` and `timing_action` on the video CSV the pipeline uses
(`find_video_csv_path`, bottom camera). Takes a few seconds per session;
cached in `timing_scan.csv`. `timing_action` gives the strings of the library's
`timing_action` (deprecated in 0.2.0, then removed): `refuse: <check>` for an
excluded camera, otherwise how it is corrected.

In [ ]:
def timing_action(checks):
    """``use harp as written``, ``fix glitches``, ``re-index`` or ``refuse: <check>``."""
    verdict = vtq.timing_verdict(checks)
    if verdict != "use":
        return verdict.replace("exclude: ", "refuse: ")
    passed = dict(zip(checks["check"], checks["passed"]))
    if not passed["no_frames_lost"]:
        return "re-index"
    if not passed["harp_has_no_glitches"]:
        return "fix glitches"
    return "use harp as written"


if IS_CO:
    scan_path = WORK / "timing_scan.csv"
    # a scan from an older library version has no "action" column: redo it
    if scan_path.exists() and "action" in pd.read_csv(scan_path, nrows=0).columns:
        scan = pd.read_csv(scan_path)
    else:
        rows = []
        for session in pred_csvs:
            videos = find_behavior_videos_folder(str(DATA / session))
            csv_path = find_video_csv_path(videos) if videos else None
            row = {"session": session, "csv_path": str(csv_path)}
            if csv_path is None:
                row["action"] = "refuse: no video CSV"
            else:
                with open(csv_path) as f:
                    row["has_header"] = not f.readline()[0].isdigit()
                try:
                    checks = vtq.check_video_timing(vtq.load_video_timing(csv_path))
                except ValueError as e:
                    row.update({"action": "refuse: unreadable", "error": str(e)})
                else:
                    by_name = checks.set_index("check")
                    row.update({
                        "action": timing_action(checks),
                        "failed_checks": ", ".join(checks.loc[checks["passed"].eq(False), "check"]),
                        "frames_lost": by_name.loc["no_frames_lost", "count"],
                        "glitch_rows": by_name.loc["harp_has_no_glitches", "rows"],
                    })
            rows.append(row)
            print(session, row["action"], f"[{row.get('failed_checks', '')}]",
                  "(header)" if row.get("has_header") else "")
        scan = pd.DataFrame(rows)
        scan.to_csv(scan_path, index=False)
    scan["failed_checks"] = scan["failed_checks"].fillna("")
    print(scan.groupby(["action", "failed_checks"]).size().to_string())
    print(f"header-row CSVs: {int(scan['has_header'].fillna(False).sum())}")

## 5. Choose sessions

Up to `N_PER_CLASS` sessions per combination of action and failed checks,
up to `N_PER_CLASS` header-row CSVs, and `N_OK` sessions that pass every
check.

In [ ]:
if IS_CO:
    header = scan["has_header"].fillna(False).astype(bool)
    clean = scan["failed_checks"].eq("")
    chosen = pd.concat([
        scan[~header & ~clean].groupby(["action", "failed_checks"]).head(N_PER_CLASS),
        scan[header].head(N_PER_CLASS),
        scan[~header & clean].head(N_OK),
    ])
    chosen = chosen.assign(group=chosen["action"].where(~header[chosen.index], "header_row"))
    selected_preds = [pred_csvs[s] for s in chosen["session"]]
    (WORK / "pred_csv_selected.json").write_text(json.dumps(selected_preds, indent=2))
    print(chosen[["session", "group", "failed_checks", "frames_lost", "glitch_rows"]].to_string(index=False))

## 6. Run the batch with each library version

Each run is a subprocess with that clone's `src` first on `PYTHONPATH`; it
checks which library it imported before running. Output goes to
`out_old/` and `out_new/` (and `out_csv_only/` if `RUN_CSV_ONLY`), the log of
each run to `<name>.log`. `force_rerun=True`, clips off. Allow a few minutes
per session per run.

In [ ]:
RUNNER = """
import json, sys
from pathlib import Path

lib_src, out_dir, preds_json, data_root, csv_only = sys.argv[1:6]
import aind_dynamic_foraging_behavior_video_analysis as lib
assert Path(lib.__file__).resolve().is_relative_to(Path(lib_src).resolve()), lib.__file__
print("library:", lib.__file__, flush=True)

from aind_dynamic_foraging_behavior_video_analysis.kinematics.tongue_analysis import run_batch_analysis
kwargs = {"use_trigger_log": False} if csv_only == "1" else {}
run_batch_analysis(
    json.loads(Path(preds_json).read_text()), Path(data_root), out_dir,
    extract_clips=False, force_rerun=True, **kwargs,
)
"""


def run_batch(name, lib_dir, csv_only=False):
    """Run run_batch_analysis on the selected sessions with one library version."""
    runner = WORK / "run_batch.py"
    runner.write_text(RUNNER)
    env = {**os.environ, "PYTHONPATH": str(lib_dir / "src"), "MPLBACKEND": "Agg"}
    log_path = WORK / f"{name}.log"
    with open(log_path, "w") as log:
        result = subprocess.run(
            [sys.executable, str(runner), str(lib_dir / "src"), str(WORK / name),
             str(WORK / "pred_csv_selected.json"), str(DATA), "1" if csv_only else "0"],
            env=env, stdout=log, stderr=subprocess.STDOUT,
        )
    tail = log_path.read_text().splitlines()
    print(f"{name}: exit {result.returncode}; library line: {tail[0] if tail else ''}")
    if result.returncode:
        print("\n".join(tail[-30:]))

In [ ]:
if IS_CO:
    run_batch("out_old", LIB_OLD)
    run_batch("out_new", LIB_NEW)
    if RUN_CSV_ONLY:
        run_batch("out_csv_only", LIB_NEW, csv_only=True)

## 7. Compare

`compare_batch_outputs.py` prints, per session and parquet, `identical` or
what changed, and each run's `batch_error_log.txt`.

In [ ]:
def compare(old, new):
    """Run the new version's compare script; return its per-file table."""
    table_path = WORK / f"diff_{old}_vs_{new}.csv"
    result = subprocess.run(
        [sys.executable, str(LIB_NEW / "scripts/compare_batch_outputs.py"),
         str(WORK / old), str(WORK / new), "--csv", str(table_path)],
        capture_output=True, text=True,
    )
    print(result.stdout, result.stderr)
    return pd.read_csv(table_path)


if IS_CO:
    diff = compare("out_old", "out_new")

Per session: the expected outcome for its action, how many files were
identical, and a `CHECK` flag where the outcome differs from the
expectation table at the top.

In [ ]:
def expected(group):
    """Expected diff for a session's group (its action, or header_row)."""
    if group.startswith("refuse"):
        return "missing in new (refused)"
    return {
        "use harp as written": "all identical",
        "fix glitches": "time columns differ slightly",
        "re-index": "large time differences",
        "header_row": "missing in old",
    }.get(group, "review")


if IS_CO:
    per_session = (
        diff.assign(identical=diff["result"].eq("identical"))
        .groupby("session")
        .agg(files=("file", "size"), identical=("identical", "sum"),
             missing_in_old=("result", lambda r: r.eq("missing in old").sum()),
             missing_in_new=("result", lambda r: r.eq("missing in new").sum()))
        .reset_index()
        .merge(chosen[["session", "group", "failed_checks"]], on="session", how="right")
    )
    per_session["expected"] = per_session["group"].map(expected)
    expect = per_session["expected"]
    per_session["flag"] = (
        per_session["files"].isna()
        | expect.eq("review")
        | (expect.eq("all identical") & (per_session["identical"] != per_session["files"]))
        | (expect.str.startswith("missing in new") & (per_session["missing_in_new"] != per_session["files"]))
        | (expect.eq("missing in old") & (per_session["missing_in_old"] != per_session["files"]))
        | (expect.isin(["time columns differ slightly", "large time differences"])
           & (per_session["missing_in_new"] > 0))
    ).map({True: "CHECK", False: ""})
    print(per_session.to_string(index=False))

In [ ]:
if IS_CO and RUN_CSV_ONLY:
    # trigger log (default) vs CSV only: expected identical without drops
    diff_csv_only = compare("out_new", "out_csv_only")

## 8. Summary

- All `ok` sessions `identical` and no `CHECK` flags: the change only
  affects sessions whose timing needed correcting.
- Header-row sessions present only in `out_new`: the new-layout fix works.
- Glitch sessions: confirm the differing columns are time columns and the
  changes are small (µs to ~1 s on a handful of rows).

Record the outcome in the library's `VIDEO_TIMING_QC_PLAN.md` (Phase 3).